# Readiculous — ML Scenario Simulator

Self-contained notebook. No Flask running, no MySQL needed.
Loads the trained `.pkl` models directly and simulates two journeys:

- **User journey** — a reader goes from zero books to a growing library and watches suggestions evolve
- **Librarian journey** — a library starts empty, patrons read and rate books, and the library learns what to stock

Books not in the Goodreads CSV are **added to the live dataset** when a user rates them,
so the recommendation pool grows with real usage.

---
**Before running:** set `GOODREADS_CSV` below to the path of your local CSV.

In [ ]:
# ── CONFIGURE THIS ──────────────────────────────────────────────────────────
GOODREADS_CSV = "/Users/whoseunassailable/Documents/datasets/GoodReads_100k_books.csv"
MODEL_DIR     = "."   # directory containing the .pkl files (same folder as this notebook)
# ────────────────────────────────────────────────────────────────────────────

import os, re, warnings
import numpy  as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display
from collections import defaultdict

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 55)
pd.set_option("display.float_format", "{:.3f}".format)

print("Imports OK")

## Setup — Load Data and Models
Mirrors exactly what `recommender.py` does at startup, so you are testing the real inference pipeline.

In [ ]:
# ── Load and clean the CSV ───────────────────────────────────────────────────

def _isbn10_to_isbn13(isbn10):
    clean = re.sub(r"[^0-9Xx]", "", str(isbn10))
    if len(clean) != 10 or not clean[:9].isdigit():
        return None
    prefix = "978" + clean[:9]
    total  = sum((1 if i % 2 == 0 else 3) * int(d) for i, d in enumerate(prefix))
    return prefix + str((10 - total % 10) % 10)

def _clean(df):
    keep = ["author", "bookformat", "genre", "isbn", "isbn13", "pages",
            "rating", "reviews", "title", "totalratings",
            "desc", "description", "Desc", "Description"]
    df = df[[c for c in keep if c in df.columns]].copy()
    df = df.dropna(subset=["rating", "genre", "author"]).reset_index(drop=True)
    df["pages"]        = df["pages"].fillna(0).astype("int32")
    df["reviews"]      = df["reviews"].fillna(0).astype("int32")
    df["totalratings"] = df["totalratings"].fillna(0).astype("int32")
    df["rating"]       = df["rating"].astype("float32")
    is_latin = df["title"].apply(
        lambda t: isinstance(t, str) and sum(ord(c) < 128 for c in t) / len(t) >= 0.8
    )
    return df[is_latin].reset_index(drop=True)

print("Reading CSV…")
raw = pd.read_csv(GOODREADS_CSV)
if "isbn" in raw.columns:
    raw["isbn13"] = raw["isbn"].apply(_isbn10_to_isbn13)
books_df = _clean(raw)
print(f"Loaded {len(books_df):,} clean books from CSV")

DESC_COL = next((c for c in ["desc", "description", "Desc", "Description"] if c in books_df.columns), None)
ISBN_COL = next((c for c in ["isbn13", "isbn"] if c in books_df.columns), None)

In [ ]:
# ── Load models ──────────────────────────────────────────────────────────────

xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb_model.pkl"))
le_author  = joblib.load(os.path.join(MODEL_DIR, "le_author.pkl"))
le_format  = joblib.load(os.path.join(MODEL_DIR, "le_format.pkl"))
tfidf_vec  = joblib.load(os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl"))

cf_path  = os.path.join(MODEL_DIR, "cf_model.pkl")
cf_model = joblib.load(cf_path) if os.path.exists(cf_path) else None

print(f"XGBoost  : loaded ({len(xgb_model.get_booster().feature_names)} features)")
print(f"TF-IDF   : loaded ({tfidf_vec.max_features} features)")
print(f"CF (SVD) : {'loaded' if cf_model else 'NOT FOUND — CF recommendations unavailable'}")

In [ ]:
# ── Core recommendation helpers (mirrors recommender.py) ─────────────────────

def _safe_encode(encoder, series):
    known = set(encoder.classes_)
    return np.where(
        series.isin(known),
        encoder.transform(series.where(series.isin(known), encoder.classes_[0])),
        0
    )

def _build_candidates(df, genres_norm):
    df2 = df.dropna(subset=["genre"]).reset_index(drop=True)
    df2["genre_list"] = df2["genre"].str.split(",").apply(
        lambda L: [g.strip().lower() for g in L]
    )
    exploded = df2.explode("genre_list")
    cands    = exploded[exploded["genre_list"].isin(genres_norm)].drop_duplicates("title").copy()
    return df2, cands, cands.index

def _xgb_scores(df_clean, idx):
    dfc = df_clean.copy()
    dfc["genre"] = dfc["genre"].fillna("other").str.lower().str.strip()
    num = pd.DataFrame({
        "log_pages":        np.log1p(dfc["pages"]),
        "log_reviews":      np.log1p(dfc["reviews"]),
        "log_totalratings": np.log1p(dfc["totalratings"]),
        "popularity_score": dfc["rating"] * np.log1p(dfc["totalratings"]),
        "review_ratio":     dfc["reviews"] / dfc["totalratings"].replace(0, np.nan),
    }).fillna(0)
    num["author_encoded"] = _safe_encode(le_author, dfc["author"].fillna("unknown"))
    num["format_encoded"] = _safe_encode(le_format, dfc["bookformat"].fillna("unknown").str.lower().str.strip())
    genre_dummies = pd.get_dummies(dfc["genre"], prefix="genre")
    feats = pd.concat([num, genre_dummies], axis=1)
    feats = feats.reindex(columns=xgb_model.get_booster().feature_names, fill_value=0)
    return xgb_model.predict_proba(feats.loc[idx])[:, 1]

def _tfidf_scores(df_clean, idx, genres_norm):
    text_col = (df_clean[DESC_COL].fillna("").astype(str) if DESC_COL
                else df_clean["title"].fillna("").astype(str) + " " + df_clean["genre"].fillna("").astype(str))
    genre_mask  = df_clean["genre_list"].apply(lambda L: bool(genres_norm.intersection(set(L))))
    sample      = df_clean[genre_mask].nlargest(min(100, genre_mask.sum()), "rating")
    profile_vec = tfidf_vec.transform([" ".join(text_col.loc[sample.index].tolist())])
    cand_vecs   = tfidf_vec.transform(text_col.loc[idx].tolist())
    return cosine_similarity(cand_vecs, profile_vec).flatten()

def _cf_score_for_user(user_id, isbn13):
    if cf_model is None or isbn13 is None:
        return 0.5
    pred = cf_model.predict(str(user_id), str(isbn13))
    return (pred.est - 1.0) / 4.0

def _genre_cap(df, top_n):
    cap, result, counts = max(2, top_n // 3), [], {}
    for _, row in df.iterrows():
        g = str(row.get("genre", "")).split(",")[0].strip().lower()
        if counts.get(g, 0) < cap:
            result.append(row)
            counts[g] = counts.get(g, 0) + 1
    return pd.DataFrame(result) if result else pd.DataFrame(columns=df.columns)

def recommend(genres, user_id=None, interaction_count=0, top_n=8):
    genres_norm = {g.lower().strip() for g in genres}
    df_clean, cands, idx = _build_candidates(books_df, genres_norm)
    if cands.empty:
        return pd.DataFrame()

    cands["xgb_score"] = _xgb_scores(df_clean, idx)
    cands["tfidf_sim"] = _tfidf_scores(df_clean, idx, genres_norm)
    content_score       = 0.8 * cands["xgb_score"] + 0.2 * cands["tfidf_sim"]

    if interaction_count < 5 or cf_model is None or user_id is None:
        content_w, cf_w = 1.0, 0.0
    elif interaction_count < 15:
        content_w, cf_w = 0.7, 0.3
    else:
        content_w, cf_w = 0.4, 0.6

    if cf_w > 0:
        cands["cf_score"]    = [_cf_score_for_user(user_id, isbn) for isbn in cands[ISBN_COL].tolist()]
        cands["final_score"] = content_w * content_score + cf_w * cands["cf_score"]
    else:
        cands["cf_score"]    = 0.0
        cands["final_score"] = content_score

    top = cands.sort_values("final_score", ascending=False).head(top_n * 3)
    top = _genre_cap(top, top_n).head(top_n)

    out_cols = ["title", "author", "genre", "rating", "xgb_score", "tfidf_sim", "cf_score", "final_score"]
    if ISBN_COL:
        out_cols = [ISBN_COL] + out_cols
    return top[[c for c in out_cols if c in top.columns]].reset_index(drop=True)

print("Helpers ready.")

In [ ]:
# ── In-memory store + interaction helpers ────────────────────────────────────
#
# user_reads[user_id]          → list of {isbn13, title, genres, rating}
# library_members[library_id]  → set of user_ids
# genre_trends[library_id]     → {genre: accumulated_score}

user_reads      = defaultdict(list)
library_members = defaultdict(set)
genre_trends    = defaultdict(lambda: defaultdict(float))


def add_book_to_dataset(title, genre, author="Unknown", rating=None, isbn13=None):
    """
    Add a book that isn't in the Goodreads CSV to the live books_df.
    This is what happens in the real system when a user rates a book that
    Google Books API found but the Kaggle CSV didn't contain.

    Numeric fields (pages, reviews, totalratings) default to 0 — the model
    handles this gracefully via log1p(0) = 0.
    The book's rating is seeded from the first user who rates it and will
    accumulate more signal on each retrain.
    """
    global books_df

    # Don't add duplicates
    if not books_df[books_df["title"].str.lower() == title.lower()].empty:
        return

    new_row = {
        "title":        title,
        "author":       author,
        "genre":        genre,
        "rating":       float(rating) if rating else 3.0,  # seed with user's rating or neutral
        "pages":        0,
        "reviews":      0,
        "totalratings": 0,
        "bookformat":   "unknown",
        "isbn13":       isbn13,
    }
    # Pad any other columns books_df has (e.g. DESC_COL) with None
    for col in books_df.columns:
        if col not in new_row:
            new_row[col] = None

    books_df = pd.concat(
        [books_df, pd.DataFrame([new_row])],
        ignore_index=True
    )
    books_df["pages"]        = books_df["pages"].fillna(0).astype("int32")
    books_df["reviews"]      = books_df["reviews"].fillna(0).astype("int32")
    books_df["totalratings"] = books_df["totalratings"].fillna(0).astype("int32")
    books_df["rating"]       = books_df["rating"].astype("float32")

    print(f"  [+] '{title}' added to dataset (genre: {genre}) — dataset now {len(books_df):,} books")


def read_book(user_id, title, rating, library_id=None, author=None, genre=None):
    """
    Simulate a user reading and rating a book.

    If the book is in the Goodreads CSV, its metadata is used automatically.
    If not found, and genre + author are provided, the book is added to the
    live dataset so it can appear in future recommendations and be picked up
    by the next retrain cycle.
    If not found and genre is missing, it is skipped with an explanation.
    """
    match = books_df[books_df["title"].str.lower() == title.lower()]
    if match.empty:
        match = books_df[books_df["title"].str.lower().str.contains(title.lower(), na=False)]

    if match.empty:
        if not genre:
            print(f"  [!] '{title}' not in dataset. Provide genre= and author= to add it.")
            return
        add_book_to_dataset(title, genre=genre, author=author or "Unknown", rating=rating)
        match = books_df[books_df["title"].str.lower() == title.lower()]

    row    = match.iloc[0]
    isbn13 = row.get(ISBN_COL) if ISBN_COL else None
    genres = [g.strip() for g in str(row["genre"]).split(",")]

    user_reads[user_id].append({"isbn13": isbn13, "title": row["title"], "genres": genres, "rating": rating})

    if library_id:
        library_members[library_id].add(user_id)
        for g in genres:
            genre_trends[library_id][g] += rating / 5.0

    print(f"  ✓ {user_id} read '{row['title']}' (rating {rating}/5) | genres: {genres[:2]}")


def get_interaction_count(user_id):
    return len(user_reads[user_id])


def get_active_genres(user_id, initial_genres=None):
    """Start from the user's chosen genres, expand with genres from books rated >= 3."""
    genres = set(initial_genres or [])
    for entry in user_reads[user_id]:
        if entry["rating"] >= 3:
            genres.update(entry["genres"])
    return list(genres)


def library_suggest(library_id, top_n=8):
    members = library_members[library_id]
    if not members:
        return None, pd.DataFrame()

    genre_votes = defaultdict(float)
    for uid in members:
        for entry in user_reads[uid]:
            weight = entry["rating"] / 5.0
            for g in entry["genres"]:
                genre_votes[g] += weight

    for g, score in genre_trends[library_id].items():
        genre_votes[g] += score

    top_genres = sorted(genre_votes, key=genre_votes.get, reverse=True)[:5]
    return top_genres, recommend(top_genres, top_n=top_n)


def show_recs(df, label, n_interactions=None):
    print(f"\n{'─'*65}")
    tag = f"  ({n_interactions} interactions)" if n_interactions is not None else ""
    print(f"  {label}{tag}")
    print(f"{'─'*65}")
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        print("  No recommendations yet.")
        return
    cols = [c for c in ["title", "author", "genre", "rating", "xgb_score", "cf_score", "final_score"] if c in df.columns]
    display(df[cols])

print("Simulation store ready.")

---
## Dataset Expansion — Books Not in the CSV

When a user rates a book that isn't in the Goodreads CSV, it gets added to `books_df` in memory.
This means:
- It can appear in recommendations immediately (same session)
- Its rating seeds the book's initial quality score for XGBoost
- The next `retrain` run picks it up from MySQL and folds it into the trained model permanently

In the real app this flow is: user rates book → Node backend calls `resolveBook()` 
(fetches from Google Books API, saves to MySQL `books` table) → next retrain merges it in.

In [ ]:
# Demonstrate: rate a book that's not in the Goodreads CSV at all.
# Without genre it gets skipped. With genre it's added and becomes recommendable.

print("── Without genre (old behaviour — skipped) ──")
read_book("test_user", "Some Brand New Book 2024", rating=5)

print()
print("── With genre (new behaviour — added to dataset) ──")
size_before = len(books_df)
read_book("test_user", "Some Brand New Book 2024", rating=5,
          author="Jane Doe", genre="Fantasy")

print()
print(f"Dataset grew from {size_before:,} → {len(books_df):,} books")
print("The new book is now part of the candidate pool for Fantasy recommendations.")

---
# Part 1 — User Journey: Ava the Reader

We follow Ava from her first login to a rich reading history.
At each step the recommendation mode shifts based on how many books she has rated.

| Interactions | Mode | Weights |
|---|---|---|
| 0–4 | Content-only | 100% XGBoost + TF-IDF |
| 5–14 | Hybrid (light CF) | 70% content / 30% CF |
| 15+ | Hybrid (strong CF) | 40% content / 60% CF |

In [ ]:
# STEP 1 — Ava signs up and picks genres. No reads yet.

AVA        = "ava_reader"
AVA_GENRES = ["Fantasy", "Mystery"]

print("STEP 1 — Ava picks genres, no reads yet")
n = get_interaction_count(AVA)
print(f"Mode: content-only  |  interactions: {n}")

recs = recommend(AVA_GENRES, user_id=AVA, interaction_count=n)
show_recs(recs, "Ava — Cold Start", n)

In [ ]:
# STEP 2 — Ava reads her first book and rates it.
# Still under the 5-book threshold so mode stays content-only.
# Her active genre set expands to include the book's genre.

print("STEP 2 — Ava reads her first book")
read_book(AVA, "The Name of the Wind", rating=5)

n      = get_interaction_count(AVA)
genres = get_active_genres(AVA, AVA_GENRES)
print(f"Active genres now: {genres}")
print(f"Mode: content-only  |  interactions: {n}")

recs = recommend(genres, user_id=AVA, interaction_count=n)
show_recs(recs, "Ava — After 1st book", n)

In [ ]:
# STEP 3 — Ava reads 3 more books, including one not in the CSV.
# The unknown book gets added to the dataset automatically.

print("STEP 3 — Ava reads 3 more books (one not in CSV)")
read_book(AVA, "Gone Girl",   rating=4)
read_book(AVA, "The Hobbit",  rating=5)
# Suppose this is a newer book not in the Goodreads dataset:
read_book(AVA, "The Midnight Library", rating=4,
          author="Matt Haig", genre="Fantasy, Fiction")

n      = get_interaction_count(AVA)
genres = get_active_genres(AVA, AVA_GENRES)
print(f"\nActive genres now: {genres}")
print(f"Mode: content-only  |  interactions: {n}")

recs = recommend(genres, user_id=AVA, interaction_count=n)
show_recs(recs, "Ava — After 4 books (still content-only)", n)

In [ ]:
# STEP 4 — Ava reads her 5th book. CF unlocks.
# Mode shifts to hybrid 70/30. The cf_score column now appears.

print("STEP 4 — 5th book: CF threshold crossed")
read_book(AVA, "A Game of Thrones", rating=5)

n      = get_interaction_count(AVA)
genres = get_active_genres(AVA, AVA_GENRES)
mode   = "hybrid (70% content / 30% CF)" if (cf_model and n >= 5) else "content-only (no CF model)"
print(f"Mode: {mode}  |  interactions: {n}")

recs = recommend(genres, user_id=AVA, interaction_count=n)
show_recs(recs, "Ava — CF unlocked", n)

In [ ]:
# STEP 5 — Ava reads 10 more books, reaching 15 total.
# Mode shifts to hybrid 40/60 — CF now drives the ranking more than content.

print("STEP 5 — Ava builds a rich reading history (15 books total)")
more_books = [
    ("The Fellowship of the Ring", 5, None, None),
    ("Sherlock Holmes",            4, None, None),
    ("American Gods",              4, None, None),
    ("Big Little Lies",            3, None, None),
    ("Mistborn",                   5, None, None),
    ("In the Woods",               4, None, None),
    ("The Way of Kings",           5, None, None),
    ("Sharp Objects",              4, None, None),
    # Two books not in the CSV — added automatically:
    ("Tomorrow, and Tomorrow, and Tomorrow", 5, "Gabrielle Zevin",  "Literary Fiction"),
    ("Fourth Wing",                          4, "Rebecca Yarros",   "Fantasy, Romance"),
]
for title, rating, author, genre in more_books:
    read_book(AVA, title, rating=rating, author=author, genre=genre)

n      = get_interaction_count(AVA)
genres = get_active_genres(AVA, AVA_GENRES)

if cf_model and n >= 15:
    mode = "hybrid (40% content / 60% CF)"
elif cf_model and n >= 5:
    mode = "hybrid (70% content / 30% CF)"
else:
    mode = "content-only (no CF model)"

print(f"\nMode: {mode}  |  interactions: {n}")
print(f"Dataset size: {len(books_df):,} books (grew from CSV baseline)")
print(f"Active genres ({len(genres)}): {genres}")

recs = recommend(genres, user_id=AVA, interaction_count=n)
show_recs(recs, "Ava — Rich history, CF dominant", n)

In [ ]:
# How much did suggestions change between cold start and now?

cold_recs  = recommend(AVA_GENRES, interaction_count=0)
final_recs = recommend(get_active_genres(AVA, AVA_GENRES), user_id=AVA, interaction_count=get_interaction_count(AVA))

cold_set  = set(cold_recs["title"])
final_set = set(final_recs["title"])
overlap   = cold_set & final_set

print(f"Cold-start titles    : {len(cold_set)}")
print(f"Personalised titles  : {len(final_set)}")
print(f"Overlap              : {len(overlap)} in common")
print(f"New suggestions      : {final_set - cold_set}")
print()
if overlap == cold_set:
    print("NOTE: No list change — CF model has not seen this simulated user.")
    print("In the real app, /api/ml/retrain incorporates Ava's ratings and CF personalises further.")
else:
    print(f"Personalisation is working — {len(final_set - cold_set)} titles shifted.")

---
# Part 2 — Librarian Journey: Library-Based Suggestions

A library opens with no members. It cannot make stocking suggestions yet.
As patrons join that specific library and rate books they borrowed,
the library learns what its community wants and suggestions emerge.

Suggestions are driven entirely by **this library's members** — not the whole platform.
Two libraries with different member communities will produce different stocking lists.

In [ ]:
# STEP L1 — Library opens with no members. Expected: no suggestions.

LIBRARY_A = "riverside_library"

print("STEP L1 — Library opens, no members yet")
top_genres, recs = library_suggest(LIBRARY_A)
show_recs(recs, f"{LIBRARY_A} — No members yet")
if not top_genres:
    print("  → Correct: no suggestions until patrons join and read.")

In [ ]:
# STEP L2 — First patrons borrow and rate books.
# Unknown books are added to the dataset automatically.

print("STEP L2 — First patrons join Riverside Library")
read_book("patron_1", "The Name of the Wind", rating=5, library_id=LIBRARY_A)
read_book("patron_1", "Dune",                 rating=4, library_id=LIBRARY_A)
read_book("patron_2", "Foundation",           rating=5, library_id=LIBRARY_A)
read_book("patron_2", "The Hobbit",           rating=3, library_id=LIBRARY_A)
read_book("patron_3", "A Game of Thrones",    rating=4, library_id=LIBRARY_A)
# A newer title not in the Goodreads CSV:
read_book("patron_3", "Babel", rating=5, library_id=LIBRARY_A,
          author="R.F. Kuang", genre="Fantasy, Historical Fiction")

print(f"\nMembers: {len(library_members[LIBRARY_A])}  |  Dataset size: {len(books_df):,}")
top_genres, recs = library_suggest(LIBRARY_A)
print(f"Top genres from members: {top_genres}")
show_recs(recs, f"{LIBRARY_A} — First Stocking Suggestions")

In [ ]:
# STEP L3 — More patrons join. Community taste solidifies.

print("STEP L3 — More patrons join, community preference solidifies")
read_book("patron_4", "Ender's Game",           rating=5, library_id=LIBRARY_A)
read_book("patron_4", "The Martian",            rating=5, library_id=LIBRARY_A)
read_book("patron_5", "Neuromancer",            rating=4, library_id=LIBRARY_A)
read_book("patron_5", "Mistborn",               rating=3, library_id=LIBRARY_A)
read_book("patron_6", "Ready Player One",       rating=5, library_id=LIBRARY_A)
read_book("patron_6", "The Hitchhiker's Guide", rating=5, library_id=LIBRARY_A)

print(f"\nMembers: {len(library_members[LIBRARY_A])}  |  Dataset size: {len(books_df):,}")
top_genres_a, recs_a = library_suggest(LIBRARY_A)
print(f"Top genres from members: {top_genres_a}")
show_recs(recs_a, f"{LIBRARY_A} — Updated Stocking Suggestions")

In [ ]:
# STEP L4 — A second library with a completely different community.

LIBRARY_B = "westside_library"

print("STEP L4 — Westside Library: different member community")
read_book("patron_a", "Pride and Prejudice",      rating=5, library_id=LIBRARY_B)
read_book("patron_a", "Outlander",                rating=5, library_id=LIBRARY_B)
read_book("patron_b", "Gone with the Wind",       rating=5, library_id=LIBRARY_B)
read_book("patron_b", "The Bronze Horseman",      rating=4, library_id=LIBRARY_B)
read_book("patron_c", "Me Before You",            rating=4, library_id=LIBRARY_B)
read_book("patron_c", "The Pillars of the Earth", rating=4, library_id=LIBRARY_B)

print(f"\nMembers: {len(library_members[LIBRARY_B])}")
top_genres_b, recs_b = library_suggest(LIBRARY_B)
print(f"Top genres from members: {top_genres_b}")
show_recs(recs_b, f"{LIBRARY_B} — Stocking Suggestions")

In [ ]:
# Do the two libraries produce different stocking lists?

titles_a = set(recs_a["title"]) if not recs_a.empty else set()
titles_b = set(recs_b["title"]) if not recs_b.empty else set()
overlap  = titles_a & titles_b

print(f"{LIBRARY_A} top genres : {top_genres_a}")
print(f"{LIBRARY_B} top genres : {top_genres_b}")
print()
print(f"{LIBRARY_A} suggestions : {len(titles_a)} books")
print(f"{LIBRARY_B} suggestions : {len(titles_b)} books")
print(f"Overlap               : {len(overlap)} books in common")
print()
if len(overlap) == 0:
    print("Perfect separation — two libraries, completely different stocking lists.")
elif len(overlap) <= 2:
    print(f"Good separation. Shared titles: {overlap}")
else:
    print(f"Some overlap — genre cap may need tuning. Shared: {overlap}")

---
## Summary

| Check | Expected result |
|---|---|
| Cold-start user gets multiple clean books | > 1 result, no gibberish titles |
| Book not in CSV + genre provided → added to dataset | Dataset size increases |
| Book not in CSV + no genre → skipped with message | No crash, clear explanation |
| Genre set grows as user reads more | New genres appear in active set |
| CF unlocks at 5 interactions | `cf_score` column populated |
| CF dominates at 15+ interactions | `final_score` shifts toward `cf_score` |
| New library with no members → no suggestions | Empty result |
| Library suggestions reflect its members' reading | Sci-Fi members → Sci-Fi stock |
| Two libraries with different members → different lists | Low/zero overlap |

---
### The feedback loop — how new books become permanent

```
User rates a book not in CSV
  → add_book_to_dataset()  adds it to live books_df  (immediate, in-session)
  → Node backend resolveBook() saves it to MySQL books table
  → POST /api/ml/retrain  pulls MySQL books + user ratings,
     merges with Kaggle CSV, retrains XGBoost + SVD
  → POST /reload  reloads models + refreshed dataset into Flask
  → Book is now a permanent part of the model
```

### Why CF scores look the same for simulated users

The SVD model was trained on real user-book ratings from MySQL.
Simulated users (`ava_reader`, `patron_1`, …) are unknown to it,
so Surprise returns the global mean for all of them — CF scores appear identical.
This is expected. After real users rate books and a retrain runs, CF becomes genuinely personalised.